# Heart Disease Prediction

This notebook explores the Heart Disease dataset, performs data preprocessing and EDA, and prepares the data for modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score
import optuna
import joblib
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# Load the datasets
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
except FileNotFoundError:
    print("Error: Files not found. Please ensure 'train.csv' and 'test.csv' are in the same directory.")

In [3]:
# Display the first few rows of the training data
train_df.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [4]:
# Check data types and missing values
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

In [5]:
train_df.describe()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,54.136706,0.714735,3.312752,130.497433,245.011814,0.079987,0.981660,152.816763,0.273725,0.716028,1.455871,0.451040,4.618873
std,181865.479132,8.256301,0.451541,0.851615,14.975802,33.681581,0.271274,0.998783,19.112927,0.445870,0.948472,0.545192,0.798549,1.950007
min,0.000000,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,157499.750000,48.000000,0.000000,3.000000,120.000000,223.000000,0.000000,0.000000,142.000000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,314999.500000,54.000000,1.000000,4.000000,130.000000,243.000000,0.000000,0.000000,157.000000,0.000000,0.100000,1.000000,0.000000,3.000000
75%,472499.250000,60.000000,1.000000,4.000000,140.000000,269.000000,0.000000,2.000000,166.000000,1.000000,1.400000,2.000000,1.000000,7.000000
max,629999.000000,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000


## 1. Data Preprocessing

In [6]:
# Drop 'id' column as it's not a feature
if 'id' in train_df.columns:
    train_df = train_df.drop('id', axis=1)
if 'id' in test_df.columns:
    test_id = test_df['id'] # Save for submission
    test_df = test_df.drop('id', axis=1)

In [7]:
# Check for missing values
print("Missing values in Train set:")
print(train_df.isnull().sum())
print("\nMissing values in Test set:")
print(test_df.isnull().sum())

Missing values in Train set:
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
dtype: int64

Missing values in Test set:
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
dtype: int64


### Encoding

In [8]:
# Encode Target Variable
train_df['Heart Disease'] = train_df['Heart Disease'].map({'Presence': 1, 'Absence': 0})

# Verify encoding
train_df['Heart Disease'].value_counts()

Heart Disease
0    347546
1    282454
Name: count, dtype: int64

Split data to avoid Data Leakage

In [9]:
X_train_df = train_df.drop('Heart Disease',axis=1)
y_train_df = train_df['Heart Disease']
X_train, X_test, y_train, y_test = train_test_split(X_train_df,y_train_df,test_size=0.2,stratify=y_train_df,random_state=42)

In [10]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(504000, 13) (126000, 13) (504000,) (126000,)


In [11]:
X_train.astype(int,float).columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium'],
      dtype='object')

In [12]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 504000 entries, 539041 to 261178
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Age                      504000 non-null  int64  
 1   Sex                      504000 non-null  int64  
 2   Chest pain type          504000 non-null  int64  
 3   BP                       504000 non-null  int64  
 4   Cholesterol              504000 non-null  int64  
 5   FBS over 120             504000 non-null  int64  
 6   EKG results              504000 non-null  int64  
 7   Max HR                   504000 non-null  int64  
 8   Exercise angina          504000 non-null  int64  
 9   ST depression            504000 non-null  float64
 10  Slope of ST              504000 non-null  int64  
 11  Number of vessels fluro  504000 non-null  int64  
 12  Thallium                 504000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 53.8 MB


In [13]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit on training data AND transform training data
# Note: We use the cleaned data from the previous step
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the SAME scaler (do not fit on test data!)
# Note: X_test was implicitly defined in your earlier split. 
# If you haven't removed outliers from X_test, that's generally fine/preferred for testing 
# to show how the model handles 'real' messy data.
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for easier viewing (optional, but helpful)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Scaling complete.")
print("X_train_scaled mean:\n", X_train_scaled.mean().head()) # Should be close to 0
print("X_train_scaled std:\n", X_train_scaled.std().head())   # Should be close to 1

Scaling complete.
X_train_scaled mean:
 Age               -3.020653e-16
Sex               -1.201297e-16
Chest pain type   -9.891206e-17
BP                -6.496955e-16
Cholesterol        1.813012e-16
dtype: float64
X_train_scaled std:
 Age                1.000001
Sex                1.000001
Chest pain type    1.000001
BP                 1.000001
Cholesterol        1.000001
dtype: float64


In [14]:
X_train = X_train_scaled
X_test = X_test_scaled

In [15]:
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(504000, 13) (126000, 13) (504000,) (126000,)


In [16]:
# Initialize results list to store model performance
results = []

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42,max_iter=5000)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
y_prob = lr_model.predict_proba(X_test)[:, 1]  # probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)

results.append({'Model': 'Logistic Regression', 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC':roc_auc, 'Object': lr_model})

print("\n--- Logistic Regression ---")
print(classification_report(y_test, y_pred))

Training Logistic Regression...

--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.86      0.87     56491

    accuracy                           0.88    126000
   macro avg       0.88      0.88      0.88    126000
weighted avg       0.88      0.88      0.88    126000



In [19]:
from sklearn.tree import DecisionTreeClassifier

print("Training Decision Tree...")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
y_prob = dt_model.predict_proba(X_test)[:, 1]  # probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)

results.append({'Model': 'Decision Tree', 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC':roc_auc,'Object': dt_model})

print("\n--- Decision Tree ---")
print(classification_report(y_test, y_pred))

Training Decision Tree...

--- Decision Tree ---
              precision    recall  f1-score   support

           0       0.84      0.84      0.84     69509
           1       0.80      0.81      0.81     56491

    accuracy                           0.83    126000
   macro avg       0.82      0.82      0.82    126000
weighted avg       0.83      0.83      0.83    126000



In [20]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
y_prob = rf_model.predict_proba(X_test)[:, 1]  # probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)

results.append({'Model': 'Random Forest', 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC':roc_auc, 'Object': rf_model})

print("\n--- Random Forest ---")
print(classification_report(y_test, y_pred))

Training Random Forest...

--- Random Forest ---
              precision    recall  f1-score   support

           0       0.89      0.90      0.89     69509
           1       0.87      0.86      0.87     56491

    accuracy                           0.88    126000
   macro avg       0.88      0.88      0.88    126000
weighted avg       0.88      0.88      0.88    126000



In [21]:
from xgboost import XGBClassifier

print("Training XGBoost...")
# XGBoost Tuned
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=8000,
    learning_rate=0.03,
    max_depth=5,
    eval_metric='auc',
    early_stopping_rounds=100,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],verbose=False)
y_pred = xgb_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
y_prob = xgb_model.predict_proba(X_test)[:, 1]  # probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)

results.append({'Model': 'XGBoost', 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC':roc_auc, 'Object': xgb_model})

print("\n--- XGBoost ---")
print(classification_report(y_test, y_pred))

Training XGBoost...

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.87      0.88     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.89      0.89    126000
weighted avg       0.89      0.89      0.89    126000



In [22]:
# LightGBM
from lightgbm import LGBMClassifier

print("Training LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=8000,
    learning_rate=0.03,
    max_depth=5,
    objective='binary',
    metric='auc',
    early_stopping_rounds=100,
    random_state=42,
    verbosity=-1
)
lgbm_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
y_pred = lgbm_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
y_prob = lgbm_model.predict_proba(X_test)[:, 1]  # probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)

results.append({'Model': 'LightGBM', 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC':roc_auc, 'Object': lgbm_model})

print("\n--- LightGBM ---")
print(classification_report(y_test, y_pred))

Training LightGBM...

--- LightGBM ---
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.87      0.88     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.89      0.89    126000
weighted avg       0.89      0.89      0.89    126000



In [23]:
# CatBoost
from catboost import CatBoostClassifier

print("Training CatBoost...")
catb_model = CatBoostClassifier(
    n_estimators=8000,
    learning_rate=0.03,
    max_depth=5,
    early_stopping_rounds=400,
    eval_metric='AUC',
    random_state=42,
    verbose=False
)
catb_model.fit(X_train, y_train, eval_set=(X_test, y_test))
y_pred = catb_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
y_prob = catb_model.predict_proba(X_test)[:, 1]  # probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)

results.append({'Model': 'CatBoost', 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC_AUC':roc_auc, 'Object': catb_model})

print("\n--- CatBoost ---")
print(classification_report(y_test, y_pred))

Training CatBoost...

--- CatBoost ---
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69509
           1       0.88      0.87      0.88     56491

    accuracy                           0.89    126000
   macro avg       0.89      0.89      0.89    126000
weighted avg       0.89      0.89      0.89    126000



In [24]:
import pandas as pd

# Display Comparison
results_df = pd.DataFrame(results).sort_values(by=['ROC_AUC'], ascending=False)
display(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1','ROC_AUC']])

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
5,CatBoost,0.890389,0.884712,0.868722,0.876645,0.956302
4,LightGBM,0.889905,0.883782,0.868669,0.876161,0.956111
3,XGBoost,0.890063,0.883976,0.868829,0.876337,0.956102
0,Logistic Regression,0.884683,0.881443,0.858225,0.869679,0.951547
2,Random Forest,0.881024,0.873645,0.858845,0.866182,0.947484
1,Decision Tree,0.826040,0.804642,0.808217,0.806426,0.824371


In [25]:
# Select Best Model
best_model_name = results_df.iloc[0]['Model']
print(f"Best Model optimized for Recall: {best_model_name}")

Best Model optimized for Recall: CatBoost


In [32]:
# Load test data
test_df = pd.read_csv('test.csv')
test_id = test_df['id']
test_features = test_df.drop('id', axis=1)
test_features_scaled = scaler.fit_transform(test_features)

# Predict probabilities using your trained CatBoost model
test_probs = catb_model.predict_proba(test_features)[:, 1]

# Create submission
submission = pd.DataFrame({
    'id': test_id,
    'Heart Disease': test_probs
})

submission.to_csv('submission.csv', index=False)
print("✅ submission.csv saved!")
submission.head()

✅ submission.csv saved!


,id,Heart Disease
0,630000,0.934757
1,630001,0.265710
2,630002,0.919907
3,630003,0.586702
4,630004,0.672944
